# Prerequisites

In [1]:
# get data for labs (saved in the mounted volume, next to the notebook)
!wget -nc -O /home/jovyan/work/around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

File ‘/home/jovyan/work/around_the_world_in_80_days.txt’ already there; not retrieving.


# 1. Word Count

Instructions:
For each cell marked "double-click and add explanation here" please answer the question in your own words.
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Defind the rdd
rdd = sc.textFile('/home/jovyan/work/around_the_world_in_80_days.txt')

In [4]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [5]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

`words` alone (without `.collect()`) does not print the actual words.
Spark transformations like `flatMap` are **lazy**: calling `flatMap`
only builds a transformation plan (a DAG), it does not read the file
or compute anything yet. So `words` just prints the RDD object's
representation, something like `PythonRDD[3] at RDD at
PythonRDD.scala:53` — an id, the RDD type, and where it was created
internally. No data has actually been read or split at this point.

Breaking down `PythonRDD[3] at RDD at PythonRDD.scala:53`: `[3]` is
the RDD's internal id in the DAG (each transformation creates a new
numbered RDD), `PythonRDD` is the RDD subclass used to run Python
code on the JVM-based Spark engine, and `PythonRDD.scala:53` is the
Scala source location where this RDD wrapper is defined — it's an
implementation detail, not something specific to our data.

In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'United',
 'States',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'You',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'Project',
 'Gutenberg',
 'License',
 'included',
 'with',
 'this',
 'eBook',
 'or',
 'online',
 'at',
 'www.gutenberg.org.',
 'If',
 'you',
 'are',
 'not',
 'located',
 'in',
 'the',
 'United',
 'States,',
 'you',
 'will',
 'have',
 'to',
 'check',
 'the',
 'laws',
 'of',
 'the',
 'country',
 'where',
 'you',
 'are',
 'located',
 'before',
 'using',
 'this',
 'eBook.',
 '',
 'Title:',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 'Author:',
 'Jules'

`words.collect()` is an **action**, not a transformation. Calling it
triggers Spark to actually execute the whole DAG built so far (read
the file, run `flatMap`) and pull every resulting element back to the
driver as a plain Python list. That's why this cell actually prints
the words, while the bare `words` in the previous cell only showed
the lazy RDD object with nothing computed yet.

In [8]:
# nicer print
for w in words.collect():
    print(w)

The
Project
Gutenberg
eBook
of
Around
the
World
in
Eighty
Days





This
eBook
is
for
the
use
of
anyone
anywhere
in
the
United
States
and
most
other
parts
of
the
world
at
no
cost
and
with
almost
no
restrictions
whatsoever.
You
may
copy
it,
give
... (output truncated to the first 50 words in the repository copy)



pensive.

About
half-past
seven
in
the
evening
Mr.
Fogg
sent
to
know
if
Aouda
would
receive
him,
and
in
a
few
moments
he
found
himself
alone
with
her.

Phileas
Fogg
took
a
chair,
and
sat
down
near
the
fireplace,
opposite
Aouda.
No
emotion
was
visible
on
his
... (output truncated to the first 50 words in the repository copy)


In [9]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [10]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
rdd.flatMap?

`flatMap(f)` applies a function `f` to every element of the RDD, and
where `map` would return one output per input element (so a list of
lists here, since each line splits into several words), `flatMap`
**flattens** the results into a single, unnested RDD. That's why
`rdd.flatMap(lambda lines: lines.split(' '))` turns an RDD of lines
directly into an RDD of individual words, instead of an RDD of
lists-of-words.

In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)
('Days', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('This', 1)
('eBook', 1)
('is', 1)
('for', 1)
('the', 1)
('use', 1)
('of', 1)
('anyone', 1)
('anywhere', 1)
('in', 1)
('the', 1)
('United', 1)
('States', 1)
('and', 1)
('most', 1)
('other', 1)
('parts', 1)
('of', 1)
('the', 1)
('world', 1)
('at', 1)
('no', 1)
('cost', 1)
('and', 1)
('with', 1)
('almost', 1)
('no', 1)
('restrictions', 1)
('whatsoever.', 1)
('You', 1)
('may', 1)
('copy', 1)
('it,', 1)
('give', 1)
... (output truncated to the first 50 tuples in the repository copy)



('Royalty', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('payments', 1)
('should', 1)
('be', 1)
('clearly', 1)
('marked', 1)
('as', 1)
('such', 1)
('and', 1)
('sent', 1)
('to', 1)
('the', 1)
('Project', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('Gutenberg', 1)
('Literary', 1)
('Archive', 1)
('Foundation', 1)
('at', 1)
('the', 1)
('address', 1)
('specified', 1)
('in', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('Section', 1)
('4,', 1)
('“Information', 1)
... (output truncated to the first 50 tuples in the repository copy)


**a. Raw word count.** I split every line on spaces, map each word to `(word, 1)` and add the ones up with `reduceByKey`. The result is correct but not very useful yet: `Gutenberg` and `of` are counted separately from `gutenberg` and `Of` because of the capital letters, and there is an empty string `''` with 2193 occurrences (it comes from empty lines and double spaces).

In [12]:
# a. count the occurence of each word
word_counts = (
    rdd.flatMap(lambda line: line.split(' '))
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
word_counts.take(20)

[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10),
 ('and', 1793),
 ('most', 43),
 ('other', 59),
 ('world', 30),
 ('at', 576),
 ('no', 124),
 ('cost', 9),
 ('with', 550),
 ('almost', 19),
 ('restrictions', 2)]

**b. Lower case.** After `.lower()`, words that only differed by case are merged. For example `of` goes from 1875 to 1926 because the `Of` at the start of sentences is now counted with it. The empty string is still there (2193), and the most frequent words are still function words like `of`, `and`, `for`.

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
lower_word_counts = (
    rdd.flatMap(lambda line: line.lower().split(' '))
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
lower_word_counts.take(20)

[('of', 1926),
 ('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('this', 341),
 ('for', 414),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('and', 1835),
 ('most', 45),
 ('other', 63),
 ('at', 645),
 ('no', 137),
 ('cost', 12),
 ('with', 562),
 ('almost', 19)]

**c. Stop words.** Filtering with my stop word set removes `the`, `of`, `and`, `to`... so the counts are now dominated by content words (`world`, `days`, `states`...). The empty string and tokens like `states,` or `www.gutenberg.org.` are still there, because the punctuation is still attached to the words. That is why step f is needed.

In [14]:
# c. eliminate the stop words.
STOP_WORDS_EN = {
    "the", "a", "an", "and", "or", "but", "if", "of", "at", "by", "for",
    "with", "about", "against", "between", "into", "through", "during",
    "before", "after", "above", "below", "to", "from", "up", "down",
    "in", "out", "on", "off", "over", "under", "again", "further",
    "then", "once", "here", "there", "when", "where", "why", "how",
    "all", "any", "both", "each", "few", "more", "most", "other",
    "some", "such", "no", "nor", "not", "only", "own", "same", "so",
    "than", "too", "very", "s", "t", "can", "will", "just", "don",
    "should", "now", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing",
    "would", "could", "i", "you", "he", "she", "it", "we", "they",
    "them", "his", "her", "its", "our", "their", "this", "that",
    "these", "those", "as", "which", "who", "whom", "what",
}

no_stopwords_counts = (
    rdd.flatMap(lambda line: line.lower().split(' '))
       .filter(lambda word: word not in STOP_WORDS_EN)
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
no_stopwords_counts.take(20)

[('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('cost', 12),
 ('almost', 19),
 ('restrictions', 2),
 ('give', 18),
 ('re-use', 2),
 ('license', 12),
 ('online', 4),
 ('www.gutenberg.org.', 4),
 ('states,', 7),
 ('country', 18),
 ('using', 6)]

**d. Alphabetical order.** `sortByKey()` sorts the keys as strings. Since the punctuation has not been removed yet, the first entries are the empty string and words starting with symbols like `#103]`, `$5,000)` or `(a)`, and real words only start later. It works as expected, but it shows the data is still dirty at this point.

In [15]:
# d. sort in alphabetical order
sorted_alpha = no_stopwords_counts.sortByKey()
sorted_alpha.take(20)

[('', 2193),
 ('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1),
 ('(b)', 1),
 ('(c)', 1),
 ('(does', 1),
 ('(if', 1),
 ('(japan),', 1),
 ('(or', 3),
 ('(saturday,', 1),
 ('(sort', 1),
 ('(sunday)', 1),
 ('(trademark/copyright)', 1)]

**e. Sort by frequency.** `sortBy(..., ascending=False)` on the count puts the most frequent tokens first: the empty string (2193), then `mr.` (373), `fogg` (365), `phileas` and `passepartout`. The names of the characters are already at the top, but the same word is split in several entries (`fogg` and `fogg,`, `passepartout` and `passepartout,`), so each count is too low.

In [16]:
# e. sort descending by word frequency
sorted_by_freq = no_stopwords_counts.sortBy(
    lambda pair: pair[1], ascending=False
)
sorted_by_freq.take(20)

[('', 2193),
 ('mr.', 373),
 ('fogg', 365),
 ('phileas', 250),
 ('passepartout', 239),
 ('him', 183),
 ('said', 157),
 ('one', 133),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('upon', 113),
 ('two', 98),
 ('hundred', 92),
 ('my', 91),
 ('replied', 89),
 ('himself', 87),
 ('project', 87),
 ('without', 86)]

**f. Punctuation and blank spaces.** Stripping the punctuation from both ends of each word and dropping the empty strings fixes the problems above: `fogg` and `fogg,` are merged and the `''` entry disappears. `string.punctuation` alone was not enough for this book, because the UTF-8 text also uses curly quotes (`“ ” ’`) and long dashes (`—`), so I added them and replaced the dashes by spaces. Without this, tokens like `“tankadere”` or `byron—at` stayed in the counts.

In [17]:
# f. remove punctuations and blank spaces
import re
import string

# string.punctuation only covers ASCII: the Gutenberg UTF-8 texts also
# use curly quotes, dashes and guillemets, so we add them explicitly
PUNCTUATION = string.punctuation + "\u201c\u201d\u2018\u2019\u2014\u2013\u00ab\u00bb\u2026\ufeff"


def normalize_line(line):
    # lower case, straighten apostrophes, turn dashes into spaces so
    # "byron\u2014at" becomes two words instead of one
    line = line.lower().replace("\u2019", "'")
    return re.sub("[\u2014\u2013]", " ", line)


def strip_punctuation(word):
    return word.strip(PUNCTUATION)


clean_counts = (
    rdd.flatMap(lambda line: normalize_line(line).split(' '))
       .map(strip_punctuation)
       .filter(lambda word: word != '' and word not in STOP_WORDS_EN)
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
clean_counts.take(20)

[('around', 34),
 ('world', 68),
 ('eighty', 30),
 ('days', 84),
 ('use', 22),
 ('anyone', 9),
 ('united', 27),
 ('states', 28),
 ('cost', 14),
 ('almost', 19),
 ('restrictions', 2),
 ('whatsoever', 2),
 ('give', 19),
 ('re-use', 2),
 ('license', 17),
 ('online', 4),
 ('country', 30),
 ('using', 6),
 ('title', 1),
 ('jules', 2)]

**All steps chained.** The function gives the final result: `fogg` (602), `passepartout` (404), `mr` (391), `phileas` (256), `fix` (240). The counts are higher than in step e (`fogg` was 365) because the variants with punctuation are now merged. The top words are the main characters and the verbs of dialogue (`said`, `replied`), which makes sense for this novel.

In [18]:
# All steps (a-f) chained together in a single function
def word_count_pipeline(text_rdd, stop_words=STOP_WORDS_EN,
                        sort_by="frequency", elision=False):
    def tokens(line):
        line = normalize_line(line)
        if elision:
            # French: l'homme -> homme, qu'il -> il
            line = re.sub(r"\b(?:qu|jusqu|lorsqu|puisqu|[ldjmtsnc])'",
                          " ", line)
        return line.split(' ')

    counts = (
        text_rdd.flatMap(tokens)
                .map(strip_punctuation)
                .filter(lambda word: word != '' and
                        word not in stop_words)
                .map(lambda word: (word, 1))
                .reduceByKey(lambda a, b: a + b)
    )
    if sort_by == "alphabetical":
        return counts.sortByKey()
    if sort_by == "frequency":
        return counts.sortBy(lambda pair: pair[1], ascending=False)
    return counts


result = word_count_pipeline(rdd, sort_by="frequency")
result.take(20)

[('fogg', 602),
 ('passepartout', 404),
 ('mr', 391),
 ('him', 323),
 ('phileas', 256),
 ('fix', 240),
 ('said', 194),
 ('one', 170),
 ('time', 126),
 ('aouda', 125),
 ('himself', 124),
 ('upon', 121),
 ('train', 119),
 ('master', 104),
 ('sir', 103),
 ('my', 103),
 ('two', 102),
 ('project', 99),
 ('hundred', 98),
 ('replied', 93)]

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [19]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # turn (name, age) into (name, (age, 1))
  .map(lambda x: (x[0], (x[1], 1)))
  # sum ages and counts per name
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # divide summed age by count to get the average
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


**Timing results.** I run each version once for warm-up (the first run also pays for the JVM start and reading the file) and then 5 times, and I keep the median. Both orders return the same 7292 rows (checked with an `assert`). The order "count before filter" was a bit faster (about 0.11 s vs 0.14 s), the opposite of what I expected, since filtering first should send fewer tuples to the shuffle. My explanation is that `reduceByKey` already combines values on each partition before the shuffle, and the book is small, so the difference is mostly noise. On a much bigger file, filtering before `reduceByKey` should matter more.

In [20]:
import statistics
import time


def timed(build_fn, *args, repeats=5, **kwargs):
    """Run build_fn(...).collect() `repeats` times after one warm-up run
    and return (output, median seconds)."""
    output = build_fn(*args, **kwargs).collect()  # warm-up (JVM, file cache)
    runs = []
    for _ in range(repeats):
        start = time.perf_counter()
        output = build_fn(*args, **kwargs).collect()
        runs.append(time.perf_counter() - start)
    median = statistics.median(runs)
    print(f"{build_fn.__name__}: median {median:.4f}s over {repeats} runs "
          f"(min {min(runs):.4f}s, max {max(runs):.4f}s, "
          f"{len(output)} rows)")
    return output, median


# order A: filter stop words out before counting -> fewer tuples
# ever reach reduceByKey/shuffle
def pipeline_filter_before_count(text_rdd):
    return (
        text_rdd.flatMap(lambda line: normalize_line(line).split(' '))
                .map(strip_punctuation)
                .filter(lambda w: w != '' and w not in STOP_WORDS_EN)
                .map(lambda w: (w, 1))
                .reduceByKey(lambda a, b: a + b)
    )


# order B: count everything first, then filter stop words out of
# the aggregated tuples -> more tuples go through reduceByKey
def pipeline_count_before_filter(text_rdd):
    return (
        text_rdd.flatMap(lambda line: normalize_line(line).split(' '))
                .map(lambda w: (strip_punctuation(w), 1))
                .reduceByKey(lambda a, b: a + b)
                .filter(lambda pair: pair[0] != '' and
                        pair[0] not in STOP_WORDS_EN)
    )


out_a, time_a = timed(pipeline_filter_before_count, rdd)
out_b, time_b = timed(pipeline_count_before_filter, rdd)

assert sorted(out_a) == sorted(out_b), "both orders must give same result"
faster = "filter-before-count" if time_a < time_b else "count-before-filter"
print(f"\nfilter-before-count: {time_a:.4f}s")
print(f"count-before-filter: {time_b:.4f}s")
print(f"Fastest order: {faster} ({max(time_a, time_b) / min(time_a, time_b):.2f}x)")

pipeline_filter_before_count: median 0.1431s over 5 runs (min 0.1028s, max 0.1796s, 7292 rows)


pipeline_count_before_filter: median 0.1139s over 5 runs (min 0.1023s, max 0.1358s, 7292 rows)

filter-before-count: 0.1431s
count-before-filter: 0.1139s
Fastest order: count-before-filter (1.26x)


## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

**English vs French.** The French text has more unique words (9800 vs 7292), which makes sense because French has more verb and adjective forms (gender, plural, conjugation). The main characters are the same in both top 20 (`fogg`, `passepartout`, `phileas`, `fix`, `aouda`, `mr`), which are the only words shared. The rest of the French top 20 is made of verbs and adverbs (`avait`, `répondit`, `bien`). `the` and `of` also appear in the French list: they come from the English Project Gutenberg license at the start and end of the file, not from the book.

In [21]:
# download the French version of the same book
!wget -nc -O /home/jovyan/work/le_tour_du_monde_en_80_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

rdd_fr = sc.textFile('/home/jovyan/work/le_tour_du_monde_en_80_jours.txt')

STOP_WORDS_FR = {
    "le", "la", "les", "un", "une", "des", "de", "du", "au", "aux",
    "et", "ou", "mais", "donc", "or", "ni", "car", "que", "qui",
    "quoi", "dont", "ce", "cet", "cette", "ces", "il", "elle", "ils",
    "elles", "on", "je", "tu", "nous", "vous", "se", "sa", "son",
    "ses", "leur", "leurs", "mon", "ma", "mes", "ton", "ta", "tes",
    "notre", "nos", "votre", "vos", "en", "y", "dans", "sur", "sous",
    "avec", "sans", "pour", "par", "vers", "chez", "entre", "pas",
    "plus", "moins", "tres", "très", "est", "sont", "etait", "était",
    "etre", "être", "avoir", "a", "ai", "as", "avons", "avez", "ont",
    "si", "comme", "ne", "n", "l", "d", "qu", "à", "au", "tout", "toute", "tous", "toutes", "meme", "même",
}

# reuse the same chained pipeline from section 1, only the stop
# words differ between languages
counts_en = word_count_pipeline(rdd, stop_words=STOP_WORDS_EN,
                                sort_by="frequency")
counts_fr = word_count_pipeline(rdd_fr, stop_words=STOP_WORDS_FR,
                                sort_by="frequency", elision=True)

print("Top 20 English words:")
for word, count in counts_en.take(20):
    print(f"  {word}: {count}")

print("\nTop 20 French words:")
for word, count in counts_fr.take(20):
    print(f"  {word}: {count}")

print(f"\nUnique words (EN): {counts_en.count()}")
print(f"Unique words (FR): {counts_fr.count()}")

# words shared by both top-20 lists (proper nouns, numbers, etc.)
top20_en = {w for w, _ in counts_en.take(20)}
top20_fr = {w for w, _ in counts_fr.take(20)}
print(f"\nShared in both top-20: {top20_en & top20_fr}")

--2026-09-24 08:56:53--  https://www.gutenberg.org/ebooks/46541.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... 

connected.


HTTP request sent, awaiting response... 

302 Found
Location: http://www.gutenberg.org/cache/epub/46541/pg46541.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-24 08:56:53--  https://www.gutenberg.org/cache/epub/46541/pg46541.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 

200 OK
Length: 472731 (462K) [text/plain]
Saving to: ‘/home/jovyan/work/le_tour_du_monde_en_80_jours.txt’


          /home/jov   0%[                    ]       0  --.-KB/s               

...


        /home/jovya  43%[=======>            ] 201.89K   471KB/s               


/home/jovyan/work/l 100%[===================>] 461.65K   742KB/s    in 0.6s    

2026-09-24 08:56:54 (742 KB/s) - ‘/home/jovyan/work/le_tour_du_monde_en_80_jours.txt’ saved [472731/472731]



Top 20 English words:
  fogg: 602
  passepartout: 404
  mr: 391
  him: 323
  phileas: 256
  fix: 240
  said: 194
  one: 170
  time: 126
  aouda: 125
  himself: 124
  upon: 121
  train: 119
  master: 104
  sir: 103
  my: 103
  two: 102
  project: 99
  hundred: 98
  replied: 93

Top 20 French words:


  fogg: 689
  passepartout: 460
  avait: 357
  lui: 335
  phileas: 332
  mr: 287
  fix: 287
  heures: 243
  répondit: 215
  bien: 194
  the: 190
  deux: 147
  monsieur: 145
  dit: 138
  aouda: 136
  après: 132
  mrs: 131
  quelques: 129
  of: 123
  maître: 123

Unique words (EN): 7292
Unique words (FR): 9800



Shared in both top-20: {'aouda', 'fix', 'mr', 'fogg', 'passepartout', 'phileas'}
